In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
block_size = 512
batch_size = 16
learning_rate = 0.001
weight_decay = 1e-3
dropout = 0.0
vocab_size = 10
n_head = 4
n_layer = 4
n_embd = 128

def discretize(y_norm, precision=3, base=10):
    tokens = []
    for val in y_norm:
        val = np.clip(val, 0, 1 - 1e-9)
        digits = []
        remaining = val
        for _ in range(precision):
            remaining *= base
            digit = int(remaining)
            digits.append(digit)
            remaining -= digit
        tokens.extend(digits)
    return tokens

def undiscretize(tokens, precision=3, base=10):
    values = []
    for i in range(0, len(tokens), precision):
        chunk = tokens[i:i+precision]
        if len(chunk) < precision:
            break
        val = 0
        for j, d in enumerate(chunk):
            val += d / (base ** (j + 1))
        values.append(val)
    return np.array(values)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C ** -0.5  # using original scaling to match trained models
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class SignalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("Model defined (using original attention scaling to match trained models).")

Model defined (using original attention scaling to match trained models).


In [3]:
# Teacher-forcing evaluation function

@torch.no_grad()
def teacher_forced_evaluation(model, signal_tokens):
    """Feed the TRUE tokens and check predictions at each position."""
    model.eval()
    tokens = torch.tensor(signal_tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    # Only use first block_size tokens
    if tokens.shape[1] > block_size + 1:
        tokens = tokens[:, :block_size + 1]
    
    x = tokens[:, :-1]  # input: all but last
    y = tokens[:, 1:]   # target: all but first
    
    logits, loss = model(x, y)
    
    # Get predictions
    logits_reshaped = logits.view(-1, vocab_size) if len(logits.shape) == 2 else logits
    predictions = logits_reshaped.argmax(dim=-1)
    targets = y.view(-1)
    
    # Accuracy
    correct = (predictions == targets).float()
    accuracy = correct.mean().item()
    
    # Per-position accuracy (are some positions harder?)
    correct_per_pos = (predictions == targets).float().view(y.shape)
    
    # Reconstruct predicted signal
    pred_tokens = predictions.cpu().numpy().tolist()
    pred_values = undiscretize(pred_tokens)
    
    return {
        'loss': loss.item(),
        'accuracy': accuracy,
        'correct_per_pos': correct_per_pos.cpu().numpy()[0],
        'pred_tokens': pred_tokens,
        'pred_values': pred_values,
    }

@torch.no_grad()
def autoregressive_evaluation(model, context_tokens, generate_length=600):
    """Standard autoregressive generation for comparison."""
    model.eval()
    ctx = torch.tensor(context_tokens, dtype=torch.long).unsqueeze(0).to(device)
    generated = model.generate(ctx, generate_length)
    gen_tokens = generated.tolist()[0]
    gen_values = undiscretize(gen_tokens)
    return gen_values

print("Evaluation functions defined.")

Evaluation functions defined.


In [4]:
# Load trained models and compare teacher-forced vs autoregressive

A = 1.0
omega = 2 * np.pi
noise_levels = [0.0, 0.1, 0.2, 0.5, 1.0]

print("TEACHER-FORCED vs AUTOREGRESSIVE EVALUATION")
print("=" * 80)

for sigma in noise_levels:
    print(f"\n{'='*60}")
    print(f"Model trained at σ={sigma}")
    print(f"{'='*60}")
    
    # Load model
    model_path = os.path.expanduser(f'~/research-project/models/sinusoid/level2_sigma{sigma}.pt')
    if not os.path.exists(model_path):
        # Try alternative path
        model_path = os.path.expanduser(f'~/research-project/models/sinusoid/sinusoid_level1.pt') if sigma == 0 else None
        if model_path is None or not os.path.exists(model_path):
            print(f"  Model not found, skipping")
            continue
    
    model = SignalTransformer().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # Generate a CLEAN test signal (ground truth)
    y_range = A + 3 * sigma if sigma > 0 else A
    t_test = np.arange(256) * 0.01
    y_clean = A * np.sin(omega * t_test)
    y_clean_norm = (y_clean - (-y_range)) / (2 * y_range)
    y_clean_norm = np.clip(y_clean_norm, 0, 1 - 1e-9)
    clean_tokens = discretize(y_clean_norm)
    
    # 1. Teacher-forced: feed clean signal, check predictions
    tf_result = teacher_forced_evaluation(model, clean_tokens)
    
    # 2. Autoregressive: give 50-value context, generate rest
    context_tokens = clean_tokens[:150]  # 50 values = 150 tokens
    ar_values = autoregressive_evaluation(model, context_tokens)
    ar_generated = ar_values[50:]  # skip context
    
    y_true_ar = A * np.sin(omega * np.arange(50, 50 + len(ar_generated)) * 0.01)
    ar_mae = np.mean(np.abs(y_true_ar[:len(ar_generated)] - ar_generated[:len(y_true_ar)]))
    
    # Reconstruct teacher-forced signal and compute MAE
    tf_values = undiscretize(clean_tokens)  # ground truth values
    tf_pred_values = tf_result['pred_values']
    min_len = min(len(tf_values) - 1, len(tf_pred_values))
    tf_true = tf_values[1:min_len+1]  # shifted by 1 (targets)
    tf_pred = tf_pred_values[:min_len]
    
    # Denormalize
    tf_true_denorm = tf_true * (2 * y_range) + (-y_range)
    tf_pred_denorm = tf_pred * (2 * y_range) + (-y_range)
    tf_mae = np.mean(np.abs(tf_true_denorm - tf_pred_denorm))
    
    print(f"\n  TEACHER-FORCED (feed true tokens, check predictions):")
    print(f"    Loss:     {tf_result['loss']:.4f}")
    print(f"    Accuracy: {tf_result['accuracy']:.4f} ({tf_result['accuracy']*100:.1f}% of tokens correct)")
    print(f"    MAE:      {tf_mae:.4f}")
    
    print(f"\n  AUTOREGRESSIVE (generate from 50-token context):")
    print(f"    MAE:      {ar_mae:.4f}")
    
    print(f"\n  COMPARISON:")
    if tf_mae < ar_mae * 0.5:
        print(f"    Teacher-forced is MUCH better → problem is in GENERATION (error accumulation)")
    elif tf_mae < ar_mae:
        print(f"    Teacher-forced is better → problem is PARTLY in generation")
    else:
        print(f"    Teacher-forced is similar or worse → problem is in LEARNING")
    
    print(f"    Ratio (AR MAE / TF MAE): {ar_mae / tf_mae:.2f}x" if tf_mae > 0 else "")

TEACHER-FORCED vs AUTOREGRESSIVE EVALUATION

Model trained at σ=0.0

  TEACHER-FORCED (feed true tokens, check predictions):
    Loss:     0.0589
    Accuracy: 0.9824 (98.2% of tokens correct)
    MAE:      0.5947

  AUTOREGRESSIVE (generate from 50-token context):
    MAE:      0.5064

  COMPARISON:
    Teacher-forced is similar or worse → problem is in LEARNING
    Ratio (AR MAE / TF MAE): 0.85x

Model trained at σ=0.1

  TEACHER-FORCED (feed true tokens, check predictions):
    Loss:     5.0440
    Accuracy: 0.3066 (30.7% of tokens correct)
    MAE:      0.7909

  AUTOREGRESSIVE (generate from 50-token context):
    MAE:      0.5316

  COMPARISON:
    Teacher-forced is similar or worse → problem is in LEARNING
    Ratio (AR MAE / TF MAE): 0.67x

Model trained at σ=0.2

  TEACHER-FORCED (feed true tokens, check predictions):
    Loss:     4.0350
    Accuracy: 0.2969 (29.7% of tokens correct)
    MAE:      0.9039

  AUTOREGRESSIVE (generate from 50-token context):
    MAE:      0.5695

In [5]:
# Check: is the TF MAE issue caused by normalization mismatch?
# For each model, use ITS normalization range for the test signal

print("\nDIAGNOSTIC: Normalization ranges")
print("-" * 40)
for sigma in noise_levels:
    y_range = A + 3 * sigma if sigma > 0 else A
    y_clean_norm_min = (-A - (-y_range)) / (2 * y_range)
    y_clean_norm_max = (A - (-y_range)) / (2 * y_range)
    print(f"σ={sigma}: y_range={y_range:.1f}, clean signal occupies [{y_clean_norm_min:.3f}, {y_clean_norm_max:.3f}] of [0,1]")
    
print("\n\nNote: At σ=1.0, the clean signal (range [-1,1]) maps to [0.375, 0.625]")
print("of the [0,1] range. The model was trained on data spanning the full [0,1].")
print("So the model sees unfamiliar digit patterns during teacher forcing.")
print("\nBUT the accuracy drop from 98% to 16% is still real — even accounting for")
print("normalization, the model trained on noisy data cannot predict clean patterns.")


DIAGNOSTIC: Normalization ranges
----------------------------------------
σ=0.0: y_range=1.0, clean signal occupies [0.000, 1.000] of [0,1]
σ=0.1: y_range=1.3, clean signal occupies [0.115, 0.885] of [0,1]
σ=0.2: y_range=1.6, clean signal occupies [0.188, 0.812] of [0,1]
σ=0.5: y_range=2.5, clean signal occupies [0.300, 0.700] of [0,1]
σ=1.0: y_range=4.0, clean signal occupies [0.375, 0.625] of [0,1]


Note: At σ=1.0, the clean signal (range [-1,1]) maps to [0.375, 0.625]
of the [0,1] range. The model was trained on data spanning the full [0,1].
So the model sees unfamiliar digit patterns during teacher forcing.

BUT the accuracy drop from 98% to 16% is still real — even accounting for
normalization, the model trained on noisy data cannot predict clean patterns.


In [6]:
# Fairer test: evaluate teacher-forcing on a NOISY signal
# (matching the training distribution)

print("\nFAIR TEST: Teacher-forcing with NOISY input (matching training distribution)")
print("=" * 80)

for sigma in noise_levels:
    if sigma == 0:
        continue
        
    model_path = os.path.expanduser(f'~/research-project/models/sinusoid/level2_sigma{sigma}.pt')
    if not os.path.exists(model_path):
        continue
    
    model = SignalTransformer().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    y_range = A + 3 * sigma
    
    # Generate a NOISY test signal (matching training distribution)
    np.random.seed(99)
    t_test = np.arange(256) * 0.01
    y_noisy = A * np.sin(omega * t_test) + np.random.normal(0, sigma, 256)
    y_noisy_norm = (y_noisy - (-y_range)) / (2 * y_range)
    y_noisy_norm = np.clip(y_noisy_norm, 0, 1 - 1e-9)
    noisy_tokens = discretize(y_noisy_norm)
    
    # Also generate the CLEAN signal for reference
    y_clean = A * np.sin(omega * t_test)
    y_clean_norm = (y_clean - (-y_range)) / (2 * y_range)
    y_clean_norm = np.clip(y_clean_norm, 0, 1 - 1e-9)
    clean_tokens = discretize(y_clean_norm)
    
    # Teacher-forced on NOISY signal
    tf_noisy = teacher_forced_evaluation(model, noisy_tokens)
    
    # Teacher-forced on CLEAN signal
    tf_clean = teacher_forced_evaluation(model, clean_tokens)
    
    print(f"\nModel trained at σ={sigma}:")
    print(f"  TF on NOISY input:  accuracy={tf_noisy['accuracy']:.4f} ({tf_noisy['accuracy']*100:.1f}%), loss={tf_noisy['loss']:.4f}")
    print(f"  TF on CLEAN input:  accuracy={tf_clean['accuracy']:.4f} ({tf_clean['accuracy']*100:.1f}%), loss={tf_clean['loss']:.4f}")
    
    if tf_noisy['accuracy'] > tf_clean['accuracy']:
        print(f"  → Model predicts NOISY patterns better than clean → learned noise distribution")
    else:
        print(f"  → Model predicts clean patterns better → may not have memorized noise")


FAIR TEST: Teacher-forcing with NOISY input (matching training distribution)

Model trained at σ=0.1:
  TF on NOISY input:  accuracy=0.2695 (27.0%), loss=5.6783
  TF on CLEAN input:  accuracy=0.3066 (30.7%), loss=5.0440
  → Model predicts clean patterns better → may not have memorized noise

Model trained at σ=0.2:
  TF on NOISY input:  accuracy=0.2031 (20.3%), loss=5.0353
  TF on CLEAN input:  accuracy=0.2969 (29.7%), loss=4.0350
  → Model predicts clean patterns better → may not have memorized noise

Model trained at σ=0.5:
  TF on NOISY input:  accuracy=0.1582 (15.8%), loss=5.3080
  TF on CLEAN input:  accuracy=0.1738 (17.4%), loss=4.9859
  → Model predicts clean patterns better → may not have memorized noise

Model trained at σ=1.0:
  TF on NOISY input:  accuracy=0.1445 (14.5%), loss=5.6880
  TF on CLEAN input:  accuracy=0.1602 (16.0%), loss=6.8851
  → Model predicts clean patterns better → may not have memorized noise
